# Working with Archives at Scale

## 1. Setup OCR engine

remove the "_" in front of "_.env" file, (only keep .env) If successful the file should disappear and then run the block below.

Run this cell once in the notebook environment to set up the OCR engine.

In [ ]:
%pip install httpx==0.27.2 openai==1.55.3 --force-reinstall
from setup import extract_text

print("OCR engine is ready.")


### What happend?

The above block install the libraries for the OCR engine. 

After the setup, we imported funtion `extract_text`, which allows us to upload an image to the elm and extracted text.

(The OCR method was covered in the previous course: "OCR: From Image to Text".)


---

## 2. Find and read images or data

Read all documents in the root folder. Each folder contians scanned images of that document.

> **STEP 01**

> Set `ROOT_FOLDER` to the `test_data` folder.

In [ ]:
from pathlib import Path

# Set the root folder where the images are located
# ".." means go up one directory level from the current working directory
ROOT_FOLDER = Path("../raw/test_data")  

> **STEP 02**

> Search for all folders located in the ROOT_FOLDER

- Note that each folder is a document.
- For exampe **97376462** is a document containing 324 scanned images


In [ ]:
# iterdir() returns an iterator of all files and folders in the ROOT_FOLDER
#is.dir() checks if the path is a directory

documents = []
for folder in ROOT_FOLDER.iterdir(): 
    if folder.is_dir(): 
        documents.append(folder.name)

documents.sort()

### List comprehension version
#  doc_folder = sorted(folder.name for folder in ROOT_FOLDER.iterdir() if folder.is_dir())

print(f"Found {len(documents)} book folders:\n {documents}")

> **STEP 03**

> Create a dictionary `images_by_doc` to store the image paths for each document


In [ ]:
SUPPORTED_IMAGE_TYPES = {".jpg", ".jpeg", ".png"} # ".webp", ".gif"

# Create a dictionary to store the image paths for each document
images_by_doc = {
    document: sorted(
        str(image_path)
        for image_path in (ROOT_FOLDER / document / "image").iterdir()
        if image_path.is_file()
        and image_path.suffix.lower() in SUPPORTED_IMAGE_TYPES
    )
    for document in documents
}

print(f"Found {len(documents)} book folders: {documents}")
for document, image_paths in images_by_doc.items():
    print(f"{document}: {len(image_paths)} image(s)")

> **STEP 04**

> Preview images

> Using test as an example, preview all pages in folder "test"


In [ ]:
from IPython.display import display
from PIL import Image

image_paths = images_by_doc["test"] #image paths for a specific document
for image_path in image_paths[:]:
    print(Path(image_path).name)
    with Image.open(image_path) as image:
        display(image)

## 3. Extract text from images

The next cell calls `extract_text` for each image in the selected document and keeps each transcription in `extracted_text_by_image`, associated with its source image path. Storage is handled separately below.

Running it sends one image request per page to the OCR engine and may incur charges.

In [ ]:
# Try swap document to document_list to see if it works for all documents.

image_paths = images_by_doc["test"]
print(image_paths)

In [ ]:
MODEL = "gpt-4o-mini"
extracted_text_by_image = {}

for index, image_path in enumerate(image_paths, start=1):
    text = extract_text(image_path, model=MODEL)
    extracted_text_by_image[image_path] = text

    print(f"[{index}/{len(image_paths)}] Extracted {image_path}")

print(f"{len(extracted_text_by_image)} transcriptions in extracted_text_by_image.")

In [ ]:
print("Sample extracted text:")
print(Path(image_paths[0]).name)
print(extracted_text_by_image[image_paths[0]])
with Image.open(image_paths[0]) as image:
    display(image)

## 4. Export extracted text

Run this cell after extraction to save each transcription as a numbered -8 text file

In [ ]:
# Create the output folder if it doesn't exist
# parents=True allows creating parent directories if they don't exist, 
# exist_ok=True prevents raising an error if the folder already exists
OUTPUT_FOLDER = Path("../output") / "test"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True) 

for index, image_path in enumerate(images_by_doc["test"], start=1):
    text = extracted_text_by_image[image_path]
    output_path = OUTPUT_FOLDER / f"{Path(image_path).stem}.txt"
    output_path.write_text(text + "\n", encoding="utf-8")
    print(f"[{index}/{len(images_by_doc['test'])}] Saved {output_path.name}")

print(f"Saved {len(extracted_text_by_image)} transcriptions to {OUTPUT_FOLDER}.")